# Reading 3D city models

`pyramids-citymodels` reads CityJSON and CityGML LOD2 as `pyramids` features. The two formats are
two serializations of one data model, so both parse into one shared representation and flatten into
a `FeatureCollection` (a GeoDataFrame subclass) with one flat table: a `surface_type` column and a
`parent_id` foreign key.

This notebook uses the driver classes directly. Once a released `pyramids-core` exposes the
`pyramids.vector_drivers` registry, the same read runs through
`FeatureCollection.read_file(path, driver="cityjson")` instead.

In [ ]:
from pathlib import Path

# Importing the package runs `import pyramids` first, which activates the vendored osgeo bindings.
import pyramids_citymodels
from pyramids_citymodels.drivers.citygml import CityGmlDriver
from pyramids_citymodels.drivers.cityjson import CityJsonDriver

# Paths are relative to this notebook; the data lives in the repo-root examples/data folder.
DATA = Path("../../examples/data")
pyramids_citymodels.__version__

## CityJSON

`strategy="semantic_surfaces"` (the default) gives one row per boundary surface, tagged with its
`surface_type`. Coordinates keep their Z (2.5D).

In [ ]:
cityjson = CityJsonDriver().read(DATA / "building.city.json", strategy="semantic_surfaces")
cityjson[["city_object_id", "surface_type", "lod"]]

## CityGML LOD2

The CityGML reader reconstructs the same boundary-surface semantics from the GML, so the output
table has the same shape as the CityJSON one.

In [ ]:
citygml = CityGmlDriver().read(DATA / "building.gml", strategy="semantic_surfaces")
citygml[["city_object_id", "surface_type"]]

## Footprints

`strategy="footprint"` gives one 2D row per object — the plan-view union of its ground surfaces.
Vertical walls are non-areal in plan view, so they are excluded from the footprint.

In [ ]:
footprint = CityJsonDriver().read(DATA / "building.city.json", strategy="footprint")
print("rows:", len(footprint), "| area:", round(footprint.geometry.iloc[0].area, 3))
footprint[["city_object_id", "geometry"]]